In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def load_docs(filepath: str) -> list:
    """Load documents from JSONL file."""
    docs = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            docs.append(json.loads(line))
    return docs


def load_queries(filepath: str) -> list:
    """Load queries from JSONL file."""
    queries = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            queries.append(json.loads(line))
    return queries


def load_qrels(filepath: str) -> pd.DataFrame:
    """Load relevance judgments from CSV."""
    return pd.read_csv(filepath)


def analyze_dataset(docs: list, queries: list, qrels: pd.DataFrame) -> tuple[pd.Series, list, list]:
    """Generate comprehensive statistics about the dataset."""
    print("=" * 80)
    print("INFORMATION RETRIEVAL TEST SET ANALYSIS")
    print("=" * 80)

    # Basic counts
    print("\n📊 DATASET OVERVIEW")
    print("-" * 80)
    print(f"Total Documents in Corpus: {len(docs):,}")
    print(f"Total Queries: {len(queries):,}")
    print(f"Total Relevance Judgments: {len(qrels):,}")

    # Unique documents with relevance judgments
    unique_relevant_docs = qrels["doc_id"].nunique()
    unique_queries_with_rels = qrels["query_id"].nunique()

    print(f"\nUnique Documents with Relevance Judgments: {unique_relevant_docs:,}")
    print(f"Unique Queries with Relevance Judgments: {unique_queries_with_rels:,}")
    print(f"Percentage of Corpus that is Relevant: {(unique_relevant_docs / len(docs) * 100):.2f}%")

    # Query statistics
    print("\n📝 QUERY STATISTICS")
    print("-" * 80)

    query_rel_counts = qrels.groupby("query_id").size()

    print(f"Average Relevant Docs per Query: {query_rel_counts.mean():.2f}")
    print(f"Median Relevant Docs per Query: {query_rel_counts.median():.0f}")
    print(f"Min Relevant Docs for a Query: {query_rel_counts.min()}")
    print(f"Max Relevant Docs for a Query: {query_rel_counts.max()}")
    print(f"Std Dev of Relevant Docs: {query_rel_counts.std():.2f}")

    # Distribution of relevant documents per query
    print("\n📈 DISTRIBUTION OF RELEVANT DOCUMENTS PER QUERY")
    print("-" * 80)
    rel_dist = Counter(query_rel_counts.values)
    for num_rels, count in sorted(rel_dist.items()):
        print(f"  {num_rels} relevant doc(s): {count} queries")

    # Document text length statistics
    print("\n📄 DOCUMENT TEXT STATISTICS")
    print("-" * 80)
    doc_lengths = [len(doc["text"]) for doc in docs]
    doc_word_counts = [len(doc["text"].split()) for doc in docs]

    print(f"Average Document Length (chars): {sum(doc_lengths) / len(doc_lengths):.0f}")
    print(f"Average Document Length (words): {sum(doc_word_counts) / len(doc_word_counts):.0f}")
    print(f"Min Document Length (words): {min(doc_word_counts)}")
    print(f"Max Document Length (words): {max(doc_word_counts)}")

    # Query text length statistics
    print("\n🔍 QUERY TEXT STATISTICS")
    print("-" * 80)
    query_lengths = [len(q["text"]) for q in queries]
    query_word_counts = [len(q["text"].split()) for q in queries]

    print(f"Average Query Length (chars): {sum(query_lengths) / len(query_lengths):.0f}")
    print(f"Average Query Length (words): {sum(query_word_counts) / len(query_word_counts):.0f}")
    print(f"Min Query Length (words): {min(query_word_counts)}")
    print(f"Max Query Length (words): {max(query_word_counts)}")

    # Sparsity analysis
    print("\n⚡ SPARSITY ANALYSIS")
    print("-" * 80)
    total_possible_pairs = len(queries) * len(docs)
    sparsity = (1 - len(qrels) / total_possible_pairs) * 100
    print(f"Total Possible Query-Doc Pairs: {total_possible_pairs:,}")
    print(f"Judged Pairs: {len(qrels):,}")
    print(f"Dataset Sparsity: {sparsity:.4f}%")

    # Sample queries with their relevant documents
    print("\n🔎 SAMPLE QUERIES AND THEIR RELEVANT DOCUMENTS")
    print("-" * 80)
    sample_queries = query_rel_counts.head(5).index.tolist()

    for qid in sample_queries:
        query_text = next((q["text"] for q in queries if q["query_id"] == str(qid)), "N/A")
        rel_docs = qrels[qrels["query_id"] == qid]["doc_id"].tolist()
        print(
            f'\nQuery ID {qid}: "{query_text[:80]}..."' if len(query_text) > 80 else f'\nQuery ID {qid}: "{query_text}"'
        )
        print(f"  → {len(rel_docs)} relevant document(s): {rel_docs}")

    return query_rel_counts, doc_word_counts, query_word_counts


def create_visualizations(query_rel_counts: pd.Series, doc_word_counts: list, query_word_counts: list) -> None:
    """Create visualizations of the dataset - saves each figure separately."""
    ailens_purple = "#b637fb"
    ailens_purple_light = "#8b70c4"

    # You can also use the base color with alpha if you prefer
    base_color = ailens_purple
    green_color = "green"

    # 1. Distribution of relevant documents per query
    _, ax1 = plt.subplots(figsize=(7, 5))
    rel_dist = Counter(query_rel_counts.values)

    # Create a series for seaborn
    relevant_counts = []
    for num_docs, freq in rel_dist.items():
        relevant_counts.extend([num_docs] * freq)

    total_queries = len(relevant_counts)

    # Calculate statistics
    relevant_series = pd.Series(relevant_counts)
    mean_relevant = relevant_series.mean()
    q1_rel = relevant_series.quantile(0.25)
    q2_rel = relevant_series.quantile(0.50)  # median
    q3_rel = relevant_series.quantile(0.75)

    # Add quartile lines
    ax1.axvline(q1_rel, color=green_color, linestyle="--", linewidth=1.5, label=f"Q1 = {q1_rel:.0f}")
    ax1.axvline(q2_rel, color=green_color, linestyle=":", linewidth=2, label=f"Median = {q2_rel:.0f}")
    ax1.axvline(q3_rel, color=green_color, linestyle="--", linewidth=1.5, label=f"Q3 = {q3_rel:.0f}")

    # Add IQR shaded area (Q1 to Q3) in green
    ax1.axvspan(q1_rel, q3_rel, alpha=0.2, color=green_color, label="IQR")

    # Main histogram
    sns.histplot(
        relevant_counts,
        color=base_color,
        label=f"{total_queries} Queries, 1,706 Documents",
        kde=False,
        stat="count",
        discrete=True,
        element="bars",
        fill=True,
        alpha=0.4,
        edgecolor=base_color,
        linewidth=1.5,
        ax=ax1,
    )

    # Add mean line
    ax1.axvline(mean_relevant, color="red", linestyle="-", linewidth=2, label=f"Mean = {mean_relevant:.2f}")

    ax1.set_xlabel("Number of Relevant Documents", fontweight="bold")
    ax1.set_ylabel("Number of Queries", fontweight="bold")
    ax1.set_title("Distribution of Relevant Documents per Query", fontweight="bold")
    ax1.grid(axis="y", alpha=0.3)
    ax1.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.savefig("../../../data/fiqa/relevant_docs_distribution.png", dpi=300, bbox_inches="tight")
    print("✅ Saved: relevant_docs_distribution.png")
    plt.close()

    # 2. Document word count distribution
    _, ax2 = plt.subplots(figsize=(7, 5))
    ax2.hist(doc_word_counts, bins=30, color=ailens_purple_light, edgecolor=base_color, alpha=1.0, linewidth=1.5)
    ax2.set_xlabel("Document Length (words)", fontweight="bold")
    ax2.set_ylabel("Frequency", fontweight="bold")
    ax2.set_title("Document Length Distribution", fontweight="bold")
    ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("../../../data/fiqa/document_length_distribution.png", dpi=300, bbox_inches="tight")
    print("✅ Saved: document_length_distribution.png")
    plt.close()

    # 3. Query word count distribution
    _, ax3 = plt.subplots(figsize=(7, 5))
    ax3.hist(query_word_counts, bins=20, color=ailens_purple_light, edgecolor=base_color, alpha=1.0, linewidth=1.5)
    ax3.set_xlabel("Query Length (words)", fontweight="bold")
    ax3.set_ylabel("Frequency", fontweight="bold")
    ax3.set_title("Query Length Distribution", fontweight="bold")
    ax3.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("../../../data/fiqa/query_length_distribution.png", dpi=300, bbox_inches="tight")
    print("✅ Saved: query_length_distribution.png")
    plt.close()

    # 4. Box plot of relevant documents per query
    _, ax4 = plt.subplots(figsize=(7, 5))
    box = ax4.boxplot([query_rel_counts.to_numpy()], vert=True, patch_artist=True)
    box["boxes"][0].set_facecolor(ailens_purple_light)
    box["boxes"][0].set_edgecolor(base_color)
    box["boxes"][0].set_linewidth(1.5)
    box["boxes"][0].set_alpha(1.0)
    # Style the other box plot elements
    for element in ["whiskers", "fliers", "means", "medians", "caps"]:
        plt.setp(box[element], color=base_color, linewidth=1.5)
    ax4.set_ylabel("Number of Relevant Documents", fontweight="bold")
    ax4.set_xticklabels(["Queries"])
    ax4.set_title("Relevant Docs per Query (Box Plot)", fontweight="bold")
    ax4.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("../../../data/fiqa/relevant_docs_boxplot.png", dpi=300, bbox_inches="tight")
    print("✅ Saved: relevant_docs_boxplot.png")
    plt.close()

    print("\n✅ All visualizations saved separately!")


def create_summary_table(docs: list, queries: list, qrels: pd.DataFrame, query_rel_counts: pd.Series) -> pd.DataFrame:
    """Create a summary table as CSV."""
    summary_data = {
        "Metric": [
            "Total Documents",
            "Total Queries",
            "Total Relevance Judgments",
            "Unique Relevant Documents",
            "Queries with Judgments",
            "Avg Relevant Docs per Query",
            "Median Relevant Docs per Query",
            "Max Relevant Docs for a Query",
            "Corpus Relevance Percentage",
            "Dataset Sparsity (%)",
        ],
        "Value": [
            len(docs),
            len(queries),
            len(qrels),
            qrels["doc_id"].nunique(),
            qrels["query_id"].nunique(),
            f"{query_rel_counts.mean():.2f}",
            int(query_rel_counts.median()),
            query_rel_counts.max(),
            f"{(qrels['doc_id'].nunique() / len(docs) * 100):.2f}%",
            f"{(1 - len(qrels) / (len(queries) * len(docs))) * 100:.4f}%",
        ],
    }

    df = pd.DataFrame(summary_data)
    df.to_csv("../../../data/fiqa/ir_testset_summary.csv", index=False)
    print("✅ Summary table saved to 'ir_testset_summary.csv'")

    return df


def main() -> None:
    """Main function to run the analysis."""
    # File paths (update these to match your file locations)
    docs_file = "../../../data/fiqa/test/fiqa_docs.jsonl"
    queries_file = "../../../data/fiqa/test/fiqa_queries.jsonl"
    qrels_file = "../../../data/fiqa/test/fiqa_qrels.csv"

    # Load data
    print("Loading data...")
    docs = load_docs(docs_file)
    queries = load_queries(queries_file)
    qrels = load_qrels(qrels_file)

    # Analyze
    query_rel_counts, doc_word_counts, query_word_counts = analyze_dataset(docs, queries, qrels)

    # Create visualizations
    print("\n" + "=" * 80)
    print("GENERATING VISUALIZATIONS")
    print("=" * 80)
    create_visualizations(query_rel_counts, doc_word_counts, query_word_counts)

    # Create summary table
    print("\n" + "=" * 80)
    print("GENERATING SUMMARY TABLE")
    print("=" * 80)
    summary_df = create_summary_table(docs, queries, qrels, query_rel_counts)
    [print(f"  {row['Metric']}: {row['Value']}") for _, row in summary_df.iterrows()]

    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE!")
    print("=" * 80)
    print("\nGenerated files:")
    print("  1. ir_testset_analysis.png - Visualizations")
    print("  2. ir_testset_summary.csv - Summary statistics")


if __name__ == "__main__":
    main()